# Redrob Ranker — Demo Sandbox

Demonstrates the **Redrob Intelligent Candidate Discovery** ranker on a 50-candidate sample.
Clones the repo, installs dependencies, and runs the full pipeline — CPU only, no network calls during ranking, no GPU.

**Runtime:** ~15 seconds for 50 candidates. Full 100K run takes ~90s on CPU.

In [ ]:
# Step 1: Clone the repo
import os
if not os.path.exists('/content/redrob-ranker'):
    !git clone https://github.com/spg3098-alt/redrob-ranker.git /content/redrob-ranker
os.chdir('/content/redrob-ranker')
print('Working directory:', os.getcwd())

In [ ]:
# Step 2: Install dependencies
!pip install -q -r /content/redrob-ranker/requirements.txt

In [ ]:
# Step 3: Run the ranker on the 50-candidate demo sample
import subprocess, sys
result = subprocess.run(
    [sys.executable, '/content/redrob-ranker/rank.py',
     '--candidates', '/content/redrob-ranker/sample_candidates.jsonl',
     '--out', '/content/redrob-ranker/demo_output.csv'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Step 4: Show results
import pandas as pd
df = pd.read_csv('/content/redrob-ranker/demo_output.csv')
print(f'Ranked {len(df)} candidates')
print()
pd.set_option('display.max_colwidth', 120)
print(df[['rank', 'candidate_id', 'score', 'reasoning']].to_string(index=False))

## How it works

```
additive = 0.22*title + 0.38*skills_trust + 0.14*semantic
         + 0.10*ml_yoe_experience + 0.08*location + 0.08*eval_text_evidence

gated    = additive
         x domain_mismatch_penalty
         x consulting_only_penalty
         x notice_multiplier        # x0.65-1.00 by notice days
         x ml_yoe_gate              # x0.25/<3y, x0.60/<4y, x0.95/<5y
         x core_coverage_gate       # x0.70 if <=1/4 must-haves covered
         x self_assessment_penalty  # x0.85 if summary admits weakness

final    = gated
         x behavioral_multiplier    # x0.40-1.0 (capped for low coverage)
         x honeypot_sink            # x0.03 for impossible profiles
```

See `src/redrob_ranker/` for full source. No LLM calls, no GPU, no network during ranking.